# Memory-Split PoC — Optimal (GPT-5.6-sol) Retriever

**Question.** At a fixed parameter budget, does a model that *offloads facts* to an external retriever (**SPLIT**) beat a dense twin that stores them in-weights (**DENSE**) once retrieval is perfect?

We assume an **optimal retriever** whose golden knowledge is generated by **GPT-5.6-sol** (`openai-group/gpt-5.6-sol`) via the TrueFoundry gateway. Facts are **real Wikidata triples** (PopQA), so GPT genuinely knows them. Single-hop; the two arms share one corpus, model, budget, and init — the only difference is whether fact values carry training loss.

**Runtime:** pick a **GPU** runtime (Runtime -> Change runtime type -> GPU). Run cells top to bottom. You'll paste a TrueFoundry token when prompted.

**Caveats:** the held-out fact-QA gap is partly definitional (dense never saw those facts); the seen split is the fairer capacity comparison; the reasoning composite is likely underpowered at this toy scale.

## 1. Get the code
The PoC lives on branch `poc/optimal-retriever`. It must be pushed to the remote for this clone to work. If the repo is private, put a token in the URL: `https://<GITHUB_TOKEN>@github.com/syz2026/Memory-Split.git`. (Running locally from inside the repo? Skip this cell.)

In [ ]:
import os
REPO_URL = "https://github.com/syz2026/Memory-Split.git"  # add a token if private
BRANCH = "poc/optimal-retriever"
if not os.path.isdir("Memory-Split") and os.path.basename(os.getcwd()) != "Memory-Split":
    !git clone --branch {BRANCH} --single-branch {REPO_URL}
if os.path.isdir("Memory-Split"):
    %cd Memory-Split
!git rev-parse --abbrev-ref HEAD

In [ ]:
# Colab preinstalls torch/numpy/matplotlib/pyyaml with a CUDA-matched build.
# Install ONLY the missing runtime deps so we never reinstall (and risk
# breaking GPU) torch. `datasets` is not needed at runtime (facts are committed).
!pip install -q tiktoken openai
import torch; print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())

## 2. Persist to Google Drive (recommended for overnight runs)
Colab runtimes are ephemeral: a disconnect wipes the local disk. This cell mounts Drive and points **checkpoints + the (paid) GPT golden cache + results** at a Drive folder via `POC_PERSIST_DIR`. The corpus stays local (fast) and is rebuilt byte-identically from the same seeds, so a checkpoint resumes cleanly.

**After a disconnect:** re-run cells 1-2, then the **build** cell (fast, deterministic), then the **train** cell — it auto-resumes from the last Drive checkpoint. Skip this cell if you don't mind re-running from scratch.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['POC_PERSIST_DIR'] = '/content/drive/MyDrive/memory_split_poc'
os.makedirs(os.environ['POC_PERSIST_DIR'], exist_ok=True)
print('checkpoints + golden cache + results ->', os.environ['POC_PERSIST_DIR'])

## 3. TrueFoundry credentials
Paste your TrueFoundry token (used as `OPENAI_API_KEY`; the SDK just points at the gateway). The smoke call fails fast here if the token/model is wrong — before any training time is spent.

In [ ]:
import os, getpass
os.environ["OPENAI_API_KEY"] = getpass.getpass("TrueFoundry token: ")
os.environ["OPENAI_BASE_URL"] = "https://tfy.promptlens.trilogy.com/v1"
os.environ["POC_GPT_MODEL"] = "openai-group/gpt-5.6-sol"

import sys; sys.path.insert(0, ".")
from evals.gpt_oracle import GatewayClient
print("gateway smoke ->", GatewayClient().smoke())  # should print e.g. 'Paris'

## 4. Build the shared corpus (offline; facts are committed, stays local)

In [ ]:
!python scripts/poc_run.py --stage build

## 5. Train the matched twins (DENSE then SPLIT)
Same model, budget, and init; only the loss mask differs. Bump `--steps` for a longer overnight run (checkpoints every `--ckpt-minutes` to Drive; auto-resumes; add `--fresh` to restart).

In [ ]:
!python scripts/poc_run.py --stage train --device auto --steps 3000 --ckpt-minutes 5

## 6. Generate golden knowledge with GPT-5.6-sol (one cached call per eval item)

In [ ]:
!python scripts/poc_run.py --stage gen-golden

## 7. Evaluate + report
SPLIT @ GPT-oracle vs DENSE @ closed-book on 50 fact-QA items (seen + held-out), plus the reasoning composite and the gold-oracle upper bound.

In [ ]:
!python scripts/poc_run.py --stage eval --gold-oracle --device auto
!python scripts/poc_run.py --stage report

In [ ]:
import json, os
from IPython.display import Image, display
persist = os.environ.get('POC_PERSIST_DIR', 'data/poc')
print(json.dumps(json.load(open(f'{persist}/poc_results.json')), indent=2))
display(Image(f'{persist}/poc_figure.png'))